# 问题 7：音乐的文化影响

音乐如何在时间与环境中表达文化影响？社会/政治/技术变化（如互联网）如何在网络中留下痕迹？

本 notebook 计算几个**数据能观察到的传播指标**，并把时间拐点对照历史事件，最后讨论数据局限与因果边界。

**关键原则：区分「描述结果」与「外部原因解释」——数据只能显示变化，不能直接证明社会/技术因果。**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100

edges = pd.read_csv('output/edges.csv', dtype=str)
by_year = pd.read_csv('output/by_year.csv')

print('edges:', edges.shape)

## 1. 跨流派传播与跨年代传播

In [ ]:
edges['cross_genre'] = edges['influencer_main_genre'] != edges['follower_main_genre']
cross_rate = edges['cross_genre'].mean()

edges['inf_start'] = pd.to_numeric(edges['influencer_active_start'], errors='coerce')
edges['fol_start'] = pd.to_numeric(edges['follower_active_start'], errors='coerce')
edges['gap'] = edges['fol_start'] - edges['inf_start']
gap_dist = edges['gap'].dropna().value_counts().sort_index()

print(f'跨流派影响边占比 = {cross_rate:.3f}')
print(f'跨年代传播（follower 出道 − influencer 出道）中位数 = {edges["gap"].dropna().median():.0f} 年')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(['Within genre', 'Cross genre'],
            [edges['cross_genre'].eq(False).sum(), edges['cross_genre'].sum()],
            color=['#4C72B0', '#55A868'])
axes[0].set_ylabel('Number of edges')
axes[0].set_title('Influence within vs across genres')

axes[1].bar(gap_dist.index.astype(str), gap_dist.values, color='#C44E52')
axes[1].set_xlabel('Decades between influencer and follower debut')
axes[1].set_ylabel('Number of edges')
axes[1].set_title('Cross-decade propagation')
plt.tight_layout(); plt.show()

## 2. 时间拐点对照社会/技术事件

在特征时间序列上标注若干历史事件，观察是否出现对应变化（相关性，非因果）。

In [ ]:
events = {
    1940: 'WWII',
    1990: 'Internet popularization',
    2008: 'Financial crisis',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat, title in [(axes[0], 'duration_ms', 'Song duration (streaming era effect)'),
                         (axes[1], 'valence', 'Valence (social mood)')]:
    ax.plot(by_year['year'], by_year[feat], color='#333333')
    for yr, label in events.items():
        ax.axvline(yr, color='#C44E52', ls='--', alpha=0.7)
        ax.text(yr, ax.get_ylim()[1] * 0.95, label, rotation=90, fontsize=8, ha='right')
    ax.set_xlabel('Year'); ax.set_ylabel(feat)
    ax.set_title(title)
plt.tight_layout(); plt.show()

## 3. 讨论：文化影响与因果边界

### 数据能观察到的传播现象
- **跨流派边占比**：反映音乐风格在流派间的扩散（如 Blues→Rock、R&B→Pop/Rock 的传承）。
- **跨年代传播范围**：影响能跨越多少个十年，说明某些「奠基型」艺术家影响的持久性。
- **特征拐点**：如 2000s 后 `duration` 骤降（流媒体/TikTok 时代）、二战前后 `valence` 波动。

### 不能做的因果断言
- 数据**没有**社会事件、技术采用、政治环境的变量；时间对应 ≠ 因果。
- 「流派之间的联系」不等于「地域、语言、族群之间的文化影响」——数据只覆盖主要流派标签。
- `popularity` 是算法热度，会随时间衰减，不能代表历史流行度。

### 数据覆盖与记录方式的局限
- 影响关系来自艺术家自述 + 行业专家，存在选择性（谁被提及、谁被记住）。
- 艺术家样本集中在部分流派，年代分布失衡（早/晚期稀疏）。
- 音频特征只覆盖部分音乐维度（无歌词、旋律、和声、音频波形）。

### 更丰富数据下的研究方案（未来建议，不加入当前分析）
- 增加影响形成时间、作品级流派、歌词/结构特征 → 更可靠地定位影响发生时刻与内容。
- 增加地域、语言、族群、传播渠道、听众数据 → 才能真正测量文化影响的传播路径。